In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# 1. Cargar los datos
df = pd.read_csv('datav1.csv')

In [ ]:
# 2. Separar Features (X), Target (y) y Grupos (song_id)
X = df.drop(columns=['label', 'song_id', 'segment_type'])
y = df['label']
groups = df['song_id']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [ ]:
# 3. GroupShuffleSplit — mismo random_state que v1 para comparación justa
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=9347234)
train_idx, test_idx = next(gss.split(X, y_encoded, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

In [ ]:
# 4. Estandarización
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# 5. Definir modelos
# ─── Modelos originales ───────────────────────────────────────────────────────
svm_rbf  = SVC(kernel='rbf',    probability=True, random_state=42)
rf       = RandomForestClassifier(n_estimators=100, random_state=42)
gb       = GradientBoostingClassifier(n_estimators=100, random_state=42)

# ─── Nuevos kernels SVM ───────────────────────────────────────────────────────
# Linear: separa con una línea recta, más rápido, mejor en espacios de alta dimensión
svm_linear = SVC(kernel='linear', probability=True, random_state=42)

# Poly: captura interacciones entre features (ej. tempo × chroma)
# degree=3 es el estándar, C=1 regularización por defecto
svm_poly   = SVC(kernel='poly', degree=3, probability=True, random_state=42)

# LinearSVC: versión optimizada del kernel lineal, mucho más rápida en datasets grandes
# Necesita CalibratedClassifierCV para obtener probabilidades
linear_svc = CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=2000))

# ─── Voting Classifier (combinación de los mejores) ──────────────────────────
# Combina SVM-Linear + GB + RF: vota por mayoría suave (soft = usa probabilidades)
voting = VotingClassifier(
    estimators=[
        ('svm_linear', SVC(kernel='linear', probability=True, random_state=42)),
        ('gb',         GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('rf',         RandomForestClassifier(n_estimators=100, random_state=42))
    ],
    voting='soft'
)

models = {
    # Originales
    "SVM (RBF)": svm_rbf,
    "Random Forest": rf,
    "Gradient Boosting": gb,
    # Nuevos
    "SVM (Linear)": svm_linear,
    "SVM (Poly)": svm_poly,
    "LinearSVC": linear_svc,
    "Voting (Linear + GB + RF)": voting
}

In [ ]:
# 6. Entrenamiento, evaluación y comparación
resultados = []  # Para tabla comparativa al final

for name, model in models.items():
    print(f"\n{'='*55}")
    print(f"  Entrenando: {name}")
    print(f"{'='*55}")

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    print(f"Exactitud (Accuracy) Global: {acc:.4f}\n")
    print("Reporte de Clasificación:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    # Guardar para tabla final
    from sklearn.metrics import f1_score
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    resultados.append({'Modelo': name, 'Accuracy': round(acc, 4), 'Macro F1': round(macro_f1, 4)})

    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Matriz de Confusión — {name}')
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Predicción')
    plt.tight_layout()
    plt.show()

In [ ]:
# 7. Tabla comparativa final de todos los modelos
df_resultados = pd.DataFrame(resultados).sort_values('Macro F1', ascending=False)
df_resultados = df_resultados.reset_index(drop=True)
df_resultados.index += 1  # Ranking desde 1

print("\n" + "="*50)
print("       RANKING FINAL DE MODELOS")
print("="*50)
print(df_resultados.to_string())
print("="*50)
print("\nMejor modelo por Macro F1:", df_resultados.iloc[0]['Modelo'])